# Threshold Coverage

Check whether thresholded activations leave enough examples per neuron for downstream formula analysis.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from theoretical.activations.source import activation_path
from theoretical.activations.threshold_coverage.analysis import run

## Setup

In [ ]:
model_id = "LiquidAI/LFM2.5-1.2B-Thinking"
task_name = "snli"
layer = "model.layers.8.feed_forward"
checkpoint_name = None
alpha = 0.5
min_acts = 10
output_directory = Path("theoretical/activations/threshold_coverage/data")

finetuned_path = activation_path(
    model_id,
    task_name,
    layer,
    task_name=task_name,
    checkpoint_name=checkpoint_name,
)

## Complete coverage result

In [ ]:
result = run(
    finetuned_path,
    task_name=task_name,
    alpha=alpha,
    min_acts=min_acts,
)
result

## Construct completed output payloads

In [ ]:
report_lines = [
    "# Threshold Coverage",
    "",
    f"task: {result['sweep']['task_name']}",
    f"selected alpha: {result['selected']['alpha']:.6g}",
    f"min_acts: {result['selected']['min_acts']}",
    f"examples: {result['sweep']['num_examples']}",
    f"neurons: {result['sweep']['num_neurons']}",
    "",
    "| alpha | expected acts | kept | kept % | below min | below min % | zero | zero % |",
    "| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |",
]
for record in result["sweep"]["records"]:
    report_lines.append(
        "| "
        + " | ".join(
            (
                f"{record['alpha']:.6g}",
                f"{record['expected_activation_count']:.2f}",
                str(record["kept_count"]),
                f"{record['kept_percent']:.6f}",
                str(record["below_min_acts_count"]),
                f"{record['below_min_acts_percent']:.6f}",
                str(record["zero_activation_count"]),
                f"{record['zero_activation_percent']:.6f}",
            )
        )
        + " |"
    )

artifact_payloads = {
    "threshold_coverage.json": result,
    "threshold_coverage.md": "\n".join(report_lines) + "\n",
}

## Save completed payloads

In [ ]:
output_directory.mkdir(parents=True, exist_ok=True)
with (output_directory / "threshold_coverage.json").open("w", encoding="utf-8") as file:
    json.dump(artifact_payloads["threshold_coverage.json"], file, indent=2)
    file.write("\n")
(output_directory / "threshold_coverage.md").write_text(
    artifact_payloads["threshold_coverage.md"],
    encoding="utf-8",
)

## Coverage plots

In [ ]:
counts = np.asarray(result["selected"]["counts"])
nonzero_counts = np.asarray(result["selected"]["nonzero_counts"])
figures = {}

for name, values, title in (
    ("counts", counts, "Post-threshold activation counts"),
    ("nonzero_counts", nonzero_counts, "Post-threshold nonzero activation counts"),
):
    figure, axis = plt.subplots(figsize=(10, 6))
    if values.size:
        bin_count = max(1, min(200, np.unique(values).size))
        axis.hist(values, bins=bin_count, color="skyblue", edgecolor="black")
        axis.axvline(min_acts, color="crimson", linestyle="--", label=f"min_acts = {min_acts}")
        axis.legend()
    else:
        axis.text(0.5, 0.5, "No activation counts", ha="center", va="center")
    axis.set_title(title)
    axis.set_xlabel("active examples")
    axis.set_ylabel("neurons")
    figure.tight_layout()
    figures[name] = figure
    plt.show()

for name, figure in figures.items():
    figure.savefig(output_directory / f"{name}.png", dpi=200)